<a href="https://colab.research.google.com/github/SaketMunda/ai-agents-tutorials/blob/master/rag_agent_langchain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Goal

In this notebook we will be doing a tutorial of building a Chatbot RAG Agent using Langchain.

RAG - Retrieval Augmented Generation, it's a technique which allows the models to enhance their answering capability outside of the data they're trained on during query time by doing ingesting.

For building this Chatbot agent we will be using 2 Step RAG chain architecture which is the easiest and most commonly used method.

It uses just a single LLM call per query.

But there are some more RAG chain architectures, which we can do in different tutorials, here are those:

- 2-step RAG (current)
- Agentic RAG : LLM-powered agent decides when and how to retrieve during reasoning
- Hybrid RAG : Combines characteristics of both approaches with validation steps


## Problem Definition

We'll build an app that answers questions based on the website's content or a blog post.



## Setup

In [1]:
# install the lanchain dependencies

!pip install langchain langchain-text-splitters langchain-community bs4

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 31.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 1.9 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


Next we will be installing the components from langchain's suite of integrations.

1. Chat Model - OpenAI
2. Embeddings - OpenAI
3. Vector store - Chroma

In [2]:
!pip install -U "langchain[openai]"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 496.3/496.3 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.2/111.2 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.8/84.8 kB 6.9 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.8
    Uninstalling langchain-core-1.2.8:
      Successfully uninstalled langchain-core-1.2.8
  Attempting uninstall: langchain
    Found existing installation: langchain 1.2.8
    Uninstalling langchain-1.2.8:
      Successfully uninstalled langchain-1.2.8


In [3]:
!pip install -U "langchain-openai"

In [4]:
!pip install -qU langchain-chroma

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 57.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 75.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 64.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.4/66.4 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.0/220.0 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 3.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currentl

### Initialise the components

In [8]:
import os
import getpass
from langchain.chat_models import init_chat_model
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

# Set the API Key
if not os.environ.get("OPENAI_API_KEY"):
  os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter API key for OpenAI: ")

# initialise the chat model
model = init_chat_model("gpt-4.1")

# initialise the embedding model
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

# setup the db
vector_store = Chroma(
    collection_name = "example_collection",
    embedding_function = embeddings,
    persist_directory = "./chroma_langchain_db"
)

## 1. Indexing

Next step is, first we need index our data and make it available for search.

It has three step process:

- Load : first we need to load our data
- Split : break the documents into smaller chunks, so that it will fit into model's finite context windows
- Store : store in vector db

rag_indexing.avif



### Load the Documents

In [10]:
# loading the documents
import bs4
from langchain_community.document_loaders import WebBaseLoader

# only keep post titles, header and content from the HTML
bs4_strainer = bs4.SoupStrainer(class_=("post-title", "post-header", "post-content"))

loader = WebBaseLoader(
    web_paths = ("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs={"parse_only": bs4_strainer}
)

docs = loader.load()

assert len(docs) == 1
print(f"Total characters: {len(docs[0].page_content)}")


Total characters: 43047


In [11]:
print(docs[0].page_content[:500])



      LLM Powered Autonomous Agents
    
Date: June 23, 2023  |  Estimated Reading Time: 31 min  |  Author: Lilian Weng


Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be framed as a powerful general problem solver.
Agent System Overview#
In


### Splitting Documents

Since our loaded document is over 42k characters which is too long to fit into the context window of many models and that too in every single request.

To handle this we'll split the Document into chunks for embedding and vectore storage.

We will be using `RecursiveCharacterTextSplitter`

In [13]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    add_start_index=True
)

all_splits = text_splitter.split_documents(docs)

print(f"Split blog into {len(all_splits)} sub-documents.")

Split blog into 63 sub-documents.


### Storing Documents

Next we need to index our 63 text chunks so that we can search over them at runtime.

In [14]:
document_ids = vector_store.add_documents(documents=all_splits)

print(document_ids[:3])

['d2997d77-a8dc-40ec-96b5-b8742fb4a08b', '0f21d890-b7c9-4adc-9d8b-46784a37b2c9', '12832571-641f-45bf-9435-1c0d4bd33dfd']


This completes the Indexing portion of the pipeline. At this point we have a query-able vectore store containing the chunked contents of our blog post.

## 2. Retrieval and generation

RAG Applications commonly work as follows:

1. Retrieve : Given a user input, relevant splits are retrieves from storage using a Retriever

2. Generate : A model produces an answer using a prompt that includes both the question with the retrieved data

### RAG agents

In [15]:
from langchain.tools import tool

@tool(response_format="content_and_artifact")
def retrieve_context(query: str):
  """Retrieve information to help answer a query."""
  retrieved_docs = vector_store.similarity_search(query=query, k=2)
  serialized = "\n\n".join(
       (f"Source: {doc.metadata}\nContent: {doc.page_content}")
       for doc in retrieved_docs
      )
  return serialized, retrieved_docs


Now we can construct the agent:

In [16]:
from langchain.agents import create_agent

tools = [retrieve_context]

# If desired, specify custom instructions
prompt = (
  "You have access to a tool that retrieves context from a blog post. "
  "Use the tool to help answer user queries."
)

agent = create_agent(model, tools, system_prompt=prompt)

Let's test this out. We construct a question that would typically require an iterative sequence of retrieval steps to answer:

In [17]:
query = (
    "What is the standard method for Task Decomposition?\n\n"
    "Once you get the answer, look up common extensions of that method."
)

for event in agent.stream(
  {"messages": [{"role": "user", "content": query}]},
  stream_mode="values",
):
   event["messages"][-1].pretty_print()

================================ Human Message =================================

What is the standard method for Task Decomposition?

Once you get the answer, look up common extensions of that method.
================================== Ai Message ==================================
Tool Calls:
  retrieve_context (call_PoEsq1WCaITKLDvgp6WB5NT6)
 Call ID: call_PoEsq1WCaITKLDvgp6WB5NT6
  Args:
    query: standard method for Task Decomposition
================================= Tool Message =================================
Name: retrieve_context

Source: {'start_index': 2578, 'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/'}
Content: Task decomposition can be done (1) by LLM with simple prompting like "Steps for XYZ.\n1.", "What are the subgoals for achieving XYZ?", (2) by using task-specific instructions; e.g. "Write a story outline." for writing a novel, or (3) with human inputs.
Another quite distinct approach, LLM+P (Liu et al. 2023), involves relying on an external cla